# Лабораторная 1. Explainable AI (XAI): Wine Quality (multiclass)

## Оглавление

- I. Загрузка и подготовка данных
- II. EDA (предварительный анализ исходных данных)
- III. Feature Engineering и Препроцессинг
- IV. Обучение базовых моделей
- V. Оптимизация выбранных моделей
- VI. Итоговое сравнение и анализ ошибок
- VII. Интерпретируемость (XAI)
- VIII. Выводы (шаблон для студента)


## Импорты

Hint: запустите ячейку один раз перед началом работы, чтобы подгрузить все зависимости и зафиксировать `random_state`.


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import optuna

from catboost import CatBoostClassifier

import shap
from lime.lime_tabular import LimeTabularExplainer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")

px.defaults.template = "plotly_white"


# I. Загрузка и подготовка данных

## 1. Загрузка датасета и первичный осмотр

Hint: ниже мы загружаем CSV напрямую из UCI и кэшируем его локально, чтобы повторные запуски были быстрыми.


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

data_dir = Path("LAB_I/data")
data_dir.mkdir(parents=True, exist_ok=True)
file_path = data_dir / "winequality-red.csv"

if file_path.exists():
    df = pd.read_csv(file_path, sep=";")
else:
    df = pd.read_csv(DATA_URL, sep=";")
    df.to_csv(file_path, sep=";", index=False)

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} cols")
df.head()


In [ ]:
df.info()

## 2. Минимальные проверки качества данных

Hint: проверьте пропуски/дубликаты и базовые статистики — это поможет понять, где EDA может быть “шумной”.


In [ ]:
display(df.describe().T)

print("Missing values:")
display(df.isna().sum().to_frame("n_missing").query("n_missing > 0"))

n_dups = int(df.duplicated().sum())
print(f"Duplicates: {n_dups}")

# II. EDA (предварительный анализ исходных данных)

Hint: в этой главе анализируем “сырые” признаки до feature engineering и масштабирования.


## 1. Дисбаланс классов (Plotly)

Hint: посмотрите, насколько редкие классы могут влиять на выбор метрик и стратегию балансировки.


In [ ]:
fig = px.histogram(
    df,
    x="quality",
    color="quality",
    nbins=df["quality"].nunique(),
    title="Wine quality: распределение классов",
)
fig.update_layout(bargap=0.1)
fig.show()


## 2. Catplot-аналоги (Plotly): alcohol / volatile acidity / sulphates

Hint: сравните распределения признаков по классам — где заметнее различия и где сильнее перекрытие.


In [ ]:
eda_features = ["alcohol", "volatile acidity", "sulphates"]

for feat in eda_features:
    fig = px.box(
        df,
        x="quality",
        y=feat,
        color="quality",
        points="outliers",
        title=f"{feat} по quality (boxplot)",
    )
    fig.show()


## 3. Группировки по качеству: mean/median (таблица)

Hint: выберите несколько “опорных” признаков и сравните средние/медианы — это даст быстрые гипотезы о важности.


In [ ]:
top_features = [
    "alcohol",
    "volatile acidity",
    "sulphates",
    "density",
    "fixed acidity",
]

summary_tbl = df.groupby("quality")[top_features].agg(["mean", "median"])
summary_tbl


## 4. Компактный violinplot через melt (Plotly)

Hint: объединённый violinplot удобен для быстрого сравнения нескольких признаков; обратите внимание на масштаб осей.


In [ ]:
violin_features = ["alcohol", "volatile acidity", "sulphates", "density"]

melted = df[["quality"] + violin_features].melt(
    id_vars="quality",
    var_name="feature",
    value_name="value",
)

fig = px.violin(
    melted,
    x="feature",
    y="value",
    color="quality",
    box=True,
    points="outliers",
    title="Violinplot (melt) для выбранных признаков",
)
fig.show()


## 5. Корреляционная матрица (Plotly)

Hint: корреляции помогают увидеть “пакеты” сильно связанных признаков (потенциальная мультиколлинеарность).


In [ ]:
feature_cols_raw = [c for c in df.columns if c != "quality"]

corr = df[feature_cols_raw + ["quality"]].corr(numeric_only=True)
fig = px.imshow(
    corr,
    color_continuous_scale="Viridis",
    title="Корреляционная матрица признаков и quality",
)
fig.update_layout(height=700)
fig.show()


# III. Feature Engineering и Препроцессинг

## 1. Feature Engineering (создание новых признаков)

Hint: добавьте несколько простых “осмысленных” комбинаций исходных признаков и подумайте, какие из них могут помочь разделить классы.


In [ ]:
df_fe = df.copy()

df_fe["total_acidity"] = (
    df_fe["fixed acidity"] + df_fe["volatile acidity"] + df_fe["citric acid"]
)
df_fe["sugar_to_acid_ratio"] = df_fe["residual sugar"] / (df_fe["total_acidity"] + 1e-5)
df_fe["alcohol_density_product"] = df_fe["alcohol"] * df_fe["density"]

feature_cols = [c for c in df_fe.columns if c != "quality"]

print("Engineered features:", ["total_acidity", "sugar_to_acid_ratio", "alcohol_density_product"])
print(f"N features: {len(feature_cols)}")
df_fe[["quality"] + ["total_acidity", "sugar_to_acid_ratio", "alcohol_density_product"]].head()


## 2. Подготовка целевой переменной (LabelEncoder)

Hint: удобно перевести классы в диапазон `0..K-1` и сохранить соответствие “код → исходная метка”.


In [ ]:
y = df_fe["quality"].to_numpy()

le = LabelEncoder()
y_enc = le.fit_transform(y)

label_mapping = pd.DataFrame({"quality": le.classes_, "class_id": np.arange(len(le.classes_))})
label_mapping


## 3. Разбиение Train/Val/Test + StandardScaler

Hint: используйте стратификацию по классу и масштабируйте признаки, обучая scaler только на train.


In [ ]:
X = df_fe[feature_cols]
idx_all = df_fe.index.to_numpy()

X_train_val, X_test, y_train_val, y_test, idx_train_val, idx_test = train_test_split(
    X,
    y_enc,
    idx_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_enc,
)

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_train_val,
    y_train_val,
    idx_train_val,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Shapes:")
print("- X_train:", X_train_scaled.shape)
print("- X_val:  ", X_val_scaled.shape)
print("- X_test: ", X_test_scaled.shape)

classes = np.unique(y_enc)
print("N classes:", len(classes))


# IV. Обучение базовых моделей

## 1. Метрики для сравнения моделей (что считаем и почему)

Hint: для дисбалансных мультиклассов часто полезны `f1_macro` и ROC AUC (в one-vs-rest постановке), но интерпретировать их стоит вместе.


In [ ]:
def roc_auc_per_class(y_true: np.ndarray, y_proba: np.ndarray, classes: np.ndarray) -> pd.DataFrame:
    y_true_bin = label_binarize(y_true, classes=classes)
    aucs = roc_auc_score(y_true_bin, y_proba, average=None)
    out = pd.DataFrame({"class_id": classes, "roc_auc_ovr": aucs})
    out["quality"] = le.inverse_transform(out["class_id"].to_numpy())
    return out[["class_id", "quality", "roc_auc_ovr"]]


def evaluate_model(
    name: str,
    model,
    X_train_s: np.ndarray,
    y_train_: np.ndarray,
    X_test_s: np.ndarray,
    y_test_: np.ndarray,
    classes_: np.ndarray,
) -> dict:
    y_pred_train = model.predict(X_train_s)
    y_pred_test = model.predict(X_test_s)

    y_proba_train = model.predict_proba(X_train_s)
    y_proba_test = model.predict_proba(X_test_s)

    metrics = {
        "name": name,
        "f1_macro_train": float(f1_score(y_train_, y_pred_train, average="macro")),
        "f1_macro_test": float(f1_score(y_test_, y_pred_test, average="macro")),
        "roc_auc_macro_ovr_test": float(
            roc_auc_score(
                label_binarize(y_test_, classes=classes_),
                y_proba_test,
                average="macro",
            )
        ),
        "y_pred_test": y_pred_test,
        "y_proba_test": y_proba_test,
    }

    print(f"=== {name} | TRAIN ===")
    print(classification_report(y_train_, y_pred_train))

    print(f"=== {name} | TEST ===")
    print(classification_report(y_test_, y_pred_test))

    display(roc_auc_per_class(y_test_, y_proba_test, classes_))

    return metrics


results: dict[str, dict] = {}


## 2. RandomForestClassifier (baseline)

### 2.1 Обучение

Hint: baseline лучше делать простым и воспроизводимым (фиксируем `random_state` и используем `class_weight='balanced'`).


In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced",
)
rf_baseline.fit(X_train_scaled, y_train)


### 2.2 Оценка качества (classification_report + ROC AUC per-class)

Hint: сравните качество на train и test — это простая проверка на переобучение.


In [ ]:
results["rf_baseline"] = evaluate_model(
    "RandomForest (baseline)",
    rf_baseline,
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
    classes,
)


## 3. CatBoostClassifier (baseline)

### 3.1 Обучение

Hint: для CatBoost в учебном baseline полезно включить `eval_set` и early stopping, чтобы увидеть динамику обучения.


In [ ]:
cb_baseline = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=RANDOM_STATE,
    auto_class_weights="Balanced",
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    verbose=False,
)

cb_baseline.fit(
    X_train_scaled,
    y_train,
    eval_set=(X_val_scaled, y_val),
    early_stopping_rounds=50,
    use_best_model=True,
)


### 3.2 Оценка качества (classification_report + ROC AUC per-class)

Hint: попробуйте понять, какие классы “самые сложные” по per-class ROC AUC и поддержке (support) в отчёте.


In [ ]:
results["cb_baseline"] = evaluate_model(
    "CatBoost (baseline)",
    cb_baseline,
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
    classes,
)


## 4. Сравнение базовых моделей (Plotly)

Hint: удобно сравнить 2–3 агрегированные метрики одним графиком; не забывайте, что разные метрики могут “ранжировать” модели по-разному.


In [ ]:
baseline_compare = pd.DataFrame(
    [
        {
            "model": "RandomForest (baseline)",
            "f1_macro_train": results["rf_baseline"]["f1_macro_train"],
            "f1_macro_test": results["rf_baseline"]["f1_macro_test"],
            "roc_auc_macro_ovr_test": results["rf_baseline"]["roc_auc_macro_ovr_test"],
        },
        {
            "model": "CatBoost (baseline)",
            "f1_macro_train": results["cb_baseline"]["f1_macro_train"],
            "f1_macro_test": results["cb_baseline"]["f1_macro_test"],
            "roc_auc_macro_ovr_test": results["cb_baseline"]["roc_auc_macro_ovr_test"],
        },
    ]
)

display(baseline_compare)

fig = px.bar(
    baseline_compare.melt(id_vars="model", var_name="metric", value_name="value"),
    x="model",
    y="value",
    color="metric",
    barmode="group",
    title="Baseline: сравнение агрегированных метрик",
)
fig.show()


# V. Оптимизация выбранных моделей

Hint: ниже Optuna подбирает гиперпараметры в ограниченном бюджете (`n_trials=10`) — этого достаточно для демонстрации подхода и экономии времени.


## 1. Оптимизация RandomForest

### 1.1 Оптимизация через Optuna (CV-3)

Hint: подумайте, почему в objective логично использовать `f1_macro` и почему мы ограничиваем число запусков.


In [ ]:
def rf_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }

    model = RandomForestClassifier(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
    )
    return float(np.mean(scores))


rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=10)

print("Best RF params:", rf_study.best_params)
print("Best CV score (f1_macro):", rf_study.best_value)


### 1.2 Оценка и сравнение с baseline

Hint: сравните baseline vs tuned не только по одной метрике — улучшение может быть “локальным” по классам.


In [ ]:
rf_best = RandomForestClassifier(
    **rf_study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced",
)
rf_best.fit(X_train_scaled, y_train)

results["rf_optuna"] = evaluate_model(
    "RandomForest (optuna)",
    rf_best,
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
    classes,
)


## 2. Оптимизация CatBoost

### 2.1 Early stopping на eval_set

Hint: early stopping помогает остановить обучение до переобучения; обратите внимание на то, как выбирается “лучшая итерация”.


In [ ]:
print("Baseline best_iteration:", getattr(cb_baseline, "best_iteration_", None))
print("Baseline tree_count:", cb_baseline.tree_count_)


### 2.2 Оптимизация гиперпараметров через Optuna

Hint: в objective можно оптимизировать метрику на `val`, сохранив early stopping; важно не “подглядывать” в test.


In [ ]:
def cb_objective(trial: optuna.Trial) -> float:
    params = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 20.0, log=True),
    }

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced",
        iterations=3000,
        verbose=False,
        **params,
    )

    model.fit(
        X_train_scaled,
        y_train,
        eval_set=(X_val_scaled, y_val),
        early_stopping_rounds=50,
        use_best_model=True,
    )

    y_val_pred = model.predict(X_val_scaled)
    return float(f1_score(y_val, y_val_pred, average="macro"))


cb_study = optuna.create_study(direction="maximize")
cb_study.optimize(cb_objective, n_trials=10)

print("Best CB params:", cb_study.best_params)
print("Best val score (f1_macro):", cb_study.best_value)


### 2.3 Оценка и сравнение с baseline

Hint: зафиксируйте, какие параметры выбрала Optuna и попробуйте объяснить “интуитивно”, почему это могло улучшить `f1_macro`.


In [ ]:
cb_best = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=RANDOM_STATE,
    auto_class_weights="Balanced",
    iterations=3000,
    verbose=False,
    **cb_study.best_params,
)

cb_best.fit(
    X_train_scaled,
    y_train,
    eval_set=(X_val_scaled, y_val),
    early_stopping_rounds=50,
    use_best_model=True,
)

results["cb_optuna"] = evaluate_model(
    "CatBoost (optuna)",
    cb_best,
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
    classes,
)


# VI. Итоговое сравнение и анализ ошибок

## 1. Итоговое сравнение метрик (Plotly)

Hint: соберите результаты в одну таблицу и сравните модели по тем метрикам, которые вы считаете приоритетными для этой задачи.


In [ ]:
model_order = [
    ("rf_baseline", "RandomForest (baseline)"),
    ("cb_baseline", "CatBoost (baseline)"),
    ("rf_optuna", "RandomForest (optuna)"),
    ("cb_optuna", "CatBoost (optuna)"),
]

metrics_df = pd.DataFrame(
    [
        {
            "model": label,
            "f1_macro_train": results[key]["f1_macro_train"],
            "f1_macro_test": results[key]["f1_macro_test"],
            "roc_auc_macro_ovr_test": results[key]["roc_auc_macro_ovr_test"],
        }
        for key, label in model_order
        if key in results
    ]
)

display(metrics_df)

fig = px.bar(
    metrics_df.melt(id_vars="model", var_name="metric", value_name="value"),
    x="model",
    y="value",
    color="metric",
    barmode="group",
    title="Итоговое сравнение: агрегированные метрики",
)
fig.show()


## 2. Пересечение ошибок моделей (set-операции + примеры строк)

Hint: посмотрите на общие ошибки двух моделей — часто там оказываются “пограничные” объекты или редкие классы.


In [ ]:
y_pred_rf = results.get("rf_optuna", results["rf_baseline"])["y_pred_test"]
y_pred_cb = results.get("cb_optuna", results["cb_baseline"])["y_pred_test"]

wrong_rf = set(idx_test[(y_pred_rf != y_test)])
wrong_cb = set(idx_test[(y_pred_cb != y_test)])
wrong_both = wrong_rf & wrong_cb
wrong_union = wrong_rf | wrong_cb

share_both_in_test = len(wrong_both) / len(y_test)
share_both_in_union = (len(wrong_both) / len(wrong_union)) if len(wrong_union) else 0.0

print(f"wrong_rf:   {len(wrong_rf)}")
print(f"wrong_cb:   {len(wrong_cb)}")
print(f"wrong_both: {len(wrong_both)}")
print(f"Share wrong_both / test: {share_both_in_test:.3f}")
print(f"Share wrong_both / union: {share_both_in_union:.3f}")

pred_tbl = (
    pd.DataFrame(
        {
            "idx": idx_test,
            "y_true": le.inverse_transform(y_test),
            "pred_rf": le.inverse_transform(y_pred_rf),
            "pred_cb": le.inverse_transform(y_pred_cb),
        }
    )
    .set_index("idx")
    .sort_index()
)

k = 5
sample_idx = sorted(list(wrong_both))[:k]

examples = df_fe.loc[sample_idx, feature_cols + ["quality"]].join(
    pred_tbl.loc[sample_idx, ["y_true", "pred_rf", "pred_cb"]],
    how="left",
)
examples


# VII. Интерпретируемость (XAI)

## 1. Глобальная интерпретация (классические методы)

### 1.1 Permutation Importance (Plotly) — для RF и CatBoost

Hint: permutation importance измеряет, насколько ухудшается качество при “перемешивании” одного признака; сравните top признаков у разных моделей.


In [ ]:
def compute_permutation_importance(
    model,
    X_test_s: np.ndarray,
    y_test_: np.ndarray,
    feature_names: list[str],
    n_repeats: int = 15,
) -> pd.DataFrame:
    perm = permutation_importance(
        model,
        X_test_s,
        y_test_,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring="f1_macro",
    )
    imp = pd.DataFrame(
        {
            "feature": feature_names,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    ).sort_values("importance_mean", ascending=False)
    return imp


def plot_permutation_importance(imp_df: pd.DataFrame, title: str, top_k: int | None = None):
    view = imp_df.head(top_k) if top_k is not None else imp_df
    fig = px.bar(
        view,
        x="importance_mean",
        y="feature",
        orientation="h",
        error_x="importance_std",
        title=title,
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()


def topk_features_by_importance(imp_df: pd.DataFrame, k: int = 5) -> list[str]:
    return imp_df.head(k)["feature"].to_list()


Permutation importance для RandomForest.

Hint: обратите внимание на разницу между top-1 и top-5 (если она есть) и на стабильность (std).


In [ ]:
rf_for_xai = rf_best

rf_perm = compute_permutation_importance(rf_for_xai, X_test_scaled, y_test, feature_cols)
plot_permutation_importance(rf_perm, "Permutation importance (RF)")

rf_top5 = topk_features_by_importance(rf_perm, k=5)
rf_top5


Permutation importance для CatBoost.

Hint: сравните топ признаков CatBoost с RandomForest — совпадают ли они полностью или есть отличия?


In [ ]:
cb_for_xai = cb_best

cb_perm = compute_permutation_importance(cb_for_xai, X_test_scaled, y_test, feature_cols)
plot_permutation_importance(cb_perm, "Permutation importance (CatBoost)")

cb_top5 = topk_features_by_importance(cb_perm, k=5)
cb_top5


### 1.2 PDP (scikit-learn) — только top-5, для RF и CatBoost

Hint: PDP показывает средний эффект признака на предсказание; подумайте, почему мы ограничиваемся top-5.


In [ ]:
def plot_pdp_topk(
    model,
    X_ref: np.ndarray,
    feature_names: list[str],
    top_features: list[str],
    title: str,
):
    feature_to_idx = {f: i for i, f in enumerate(feature_names)}
    feats_idx = [feature_to_idx[f] for f in top_features]

    fig, ax = plt.subplots(figsize=(14, 8))
    PartialDependenceDisplay.from_estimator(
        model,
        X_ref,
        feats_idx,
        feature_names=feature_names,
        ax=ax,
    )
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


PDP для RandomForest (top-5 по permutation importance).


In [ ]:
plot_pdp_topk(
    rf_for_xai,
    X_train_scaled,
    feature_cols,
    rf_top5,
    title="PDP (RF): top-5 признаков",
)


PDP для CatBoost (top-5 по permutation importance).


In [ ]:
plot_pdp_topk(
    cb_for_xai,
    X_train_scaled,
    feature_cols,
    cb_top5,
    title="PDP (CatBoost): top-5 признаков",
)


## 2. Глобальная интерпретация (SHAP Summary)

### 2.1 SHAP для RandomForest (plot_cmap='viridis')

Hint: для мультикласса можно агрегировать вклад по классам (например, через среднее по |SHAP|), чтобы получить один глобальный рейтинг признаков.


In [ ]:
def make_shap_explainer(model, X_background: np.ndarray):
    return shap.TreeExplainer(model, data=X_background)


def plot_shap_summary_global(
    explainer,
    X_sample: np.ndarray,
    feature_names: list[str],
    title: str,
    plot_cmap: str = "viridis",
):
    shap_values = explainer.shap_values(X_sample)

    if isinstance(shap_values, list):
        # list[class] -> (n_samples, n_features)
        vals = np.stack(shap_values, axis=-1)
    else:
        vals = np.asarray(shap_values)

    if vals.ndim == 3:
        # (n, p, k) -> агрегируем по классам
        vals_2d = np.mean(np.abs(vals), axis=2)
    else:
        vals_2d = vals

    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        vals_2d,
        X_sample,
        feature_names=feature_names,
        show=False,
        plot_type="bar",
        cmap=plot_cmap,
    )
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
X_bg = X_train_scaled[:200]
X_shap = X_test_scaled[:300]

rf_explainer = make_shap_explainer(rf_for_xai, X_bg)
plot_shap_summary_global(
    rf_explainer,
    X_shap,
    feature_cols,
    title="SHAP summary (RF): mean(|SHAP|) по классам",
    plot_cmap="viridis",
)


### 2.2 SHAP для CatBoost (plot_cmap='viridis')

Hint: проверьте, совпадает ли топ признаков SHAP с permutation importance и PDP.


In [ ]:
cb_explainer = make_shap_explainer(cb_for_xai, X_bg)
plot_shap_summary_global(
    cb_explainer,
    X_shap,
    feature_cols,
    title="SHAP summary (CatBoost): mean(|SHAP|) по классам",
    plot_cmap="viridis",
)


## 3. Локальная интерпретация (SHAP + LIME)

### 3.1 Выбор 10 correct + 10 wrong (общий список)

Hint: сделайте единый список объектов, чтобы сравнивать локальные объяснения разных моделей на одних и тех же строках.


In [ ]:
def select_common_indices(
    idx: np.ndarray,
    y_true: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    n_each: int = 10,
) -> tuple[list[int], list[int], pd.DataFrame]:
    # Базовое правило из плана:
    # - correct: сначала "оба верны"; если не хватает, добираем "хотя бы одна верна"
    # - wrong:   сначала "оба ошиблись"; если не хватает, добираем "ошибся хотя бы один"
    both_correct = (pred_a == y_true) & (pred_b == y_true)
    at_least_one_correct = (pred_a == y_true) | (pred_b == y_true)

    both_wrong = (pred_a != y_true) & (pred_b != y_true)
    at_least_one_wrong = (pred_a != y_true) | (pred_b != y_true)

    idx_list = idx.tolist()

    def pick(mask: np.ndarray, k: int, exclude: set[int]) -> list[int]:
        out: list[int] = []
        for i, ok in zip(idx_list, mask.tolist()):
            if ok and i not in exclude:
                out.append(i)
                if len(out) >= k:
                    break
        return out

    chosen: set[int] = set()

    correct = pick(both_correct, n_each, chosen)
    chosen.update(correct)
    if len(correct) < n_each:
        correct += pick(at_least_one_correct, n_each - len(correct), chosen)
        chosen.update(correct)

    wrong = pick(both_wrong, n_each, chosen)
    chosen.update(wrong)
    if len(wrong) < n_each:
        wrong += pick(at_least_one_wrong, n_each - len(wrong), chosen)
        chosen.update(wrong)

    tbl = (
        pd.DataFrame(
            {
                "idx": idx,
                "y_true": y_true,
                "pred_rf": pred_a,
                "pred_cb": pred_b,
            }
        )
        .set_index("idx")
        .assign(
            y_true_label=lambda d: le.inverse_transform(d["y_true"].to_numpy()),
            pred_rf_label=lambda d: le.inverse_transform(d["pred_rf"].to_numpy()),
            pred_cb_label=lambda d: le.inverse_transform(d["pred_cb"].to_numpy()),
        )
    )

    return correct, wrong, tbl


# Используем tuned модели (если они есть) — это лучше соответствует дальнейшему XAI
pred_rf_test = y_pred_rf
pred_cb_test = y_pred_cb

correct_idx, wrong_idx, compare_tbl = select_common_indices(
    idx_test,
    y_test,
    pred_rf_test,
    pred_cb_test,
    n_each=10,
)

selected_idx = correct_idx + wrong_idx

print("Selected:")
print("- correct:", len(correct_idx))
print("- wrong:  ", len(wrong_idx))
print("- total:  ", len(selected_idx))

compare_tbl.loc[selected_idx, ["y_true_label", "pred_rf_label", "pred_cb_label"]]


### 3.2 SHAP heatmap для 20 объектов (оптом)

Hint: попробуйте объяснять вклад в **предсказанный класс** для каждого объекта, чтобы интерпретация была согласованной.


In [ ]:
def to_shap_values_3d(explainer, X: np.ndarray) -> np.ndarray:
    vals = explainer.shap_values(X)
    if isinstance(vals, list):
        return np.stack(vals, axis=-1)  # (n, p, k)
    arr = np.asarray(vals)
    if arr.ndim == 2:
        # бинарная/регрессия
        return arr[:, :, None]
    return arr  # ожидаем (n, p, k)


def shap_values_for_predicted_class(
    shap_3d: np.ndarray,
    pred_class: np.ndarray,
) -> np.ndarray:
    # shap_3d: (n, p, k), pred_class: (n,)
    n, p, k = shap_3d.shape
    out = np.empty((n, p), dtype=float)
    for i in range(n):
        c = int(pred_class[i])
        c = min(max(c, 0), k - 1)
        out[i, :] = shap_3d[i, :, c]
    return out


def build_shap_explanation(
    values_2d: np.ndarray,
    data_2d: np.ndarray,
    feature_names: list[str],
) -> shap.Explanation:
    return shap.Explanation(values=values_2d, data=data_2d, feature_names=feature_names)


def plot_shap_heatmap_for_selected(
    model_name: str,
    explainer,
    X_all: np.ndarray,
    pred_all: np.ndarray,
    selected_rows: list[int],
    idx_all: np.ndarray,
    feature_names: list[str],
):
    pos = np.array([np.where(idx_all == i)[0][0] for i in selected_rows], dtype=int)
    X_sel = X_all[pos]
    pred_sel = pred_all[pos]

    shap_3d = to_shap_values_3d(explainer, X_sel)
    vals_2d = shap_values_for_predicted_class(shap_3d, pred_sel)

    exp = build_shap_explanation(vals_2d, X_sel, feature_names)

    plt.figure(figsize=(12, 6))
    shap.plots.heatmap(exp, show=False)
    plt.title(f"SHAP heatmap ({model_name}) — вклад в предсказанный класс")
    plt.tight_layout()
    plt.show()


SHAP heatmap для RandomForest на выбранных 20 объектах.


In [ ]:
plot_shap_heatmap_for_selected(
    "RF",
    rf_explainer,
    X_test_scaled,
    pred_rf_test,
    selected_idx,
    idx_test,
    feature_cols,
)


SHAP heatmap для CatBoost на выбранных 20 объектах.


In [ ]:
plot_shap_heatmap_for_selected(
    "CatBoost",
    cb_explainer,
    X_test_scaled,
    pred_cb_test,
    selected_idx,
    idx_test,
    feature_cols,
)


### 3.3 SHAP waterfall: 1 correct + 1 wrong (точечно)

Hint: выберите один “очевидно правильный” и один “явно ошибочный” объект и сравните, какие признаки толкают модель к решению.


In [ ]:
def plot_shap_waterfall_one(
    model_name: str,
    explainer,
    X_all: np.ndarray,
    pred_all: np.ndarray,
    row_idx: int,
    idx_all: np.ndarray,
    feature_names: list[str],
):
    pos = int(np.where(idx_all == row_idx)[0][0])
    x = X_all[pos : pos + 1]
    pred = int(pred_all[pos])

    shap_3d = to_shap_values_3d(explainer, x)
    vals_1d = shap_values_for_predicted_class(shap_3d, np.array([pred]))[0]

    exp = shap.Explanation(values=vals_1d, data=x[0], feature_names=feature_names)

    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(exp, show=False)
    plt.title(f"SHAP waterfall ({model_name}) — idx={row_idx}, pred_class={pred}")
    plt.tight_layout()
    plt.show()


idx_correct_1 = correct_idx[0]
idx_wrong_1 = wrong_idx[0]

print("Chosen examples:")
print("- correct idx:", idx_correct_1)
print("- wrong idx:  ", idx_wrong_1)


Waterfall для RandomForest: 1 correct + 1 wrong.


In [ ]:
plot_shap_waterfall_one(
    "RF",
    rf_explainer,
    X_test_scaled,
    pred_rf_test,
    idx_correct_1,
    idx_test,
    feature_cols,
)

plot_shap_waterfall_one(
    "RF",
    rf_explainer,
    X_test_scaled,
    pred_rf_test,
    idx_wrong_1,
    idx_test,
    feature_cols,
)


Waterfall для CatBoost: 1 correct + 1 wrong.


In [ ]:
plot_shap_waterfall_one(
    "CatBoost",
    cb_explainer,
    X_test_scaled,
    pred_cb_test,
    idx_correct_1,
    idx_test,
    feature_cols,
)

plot_shap_waterfall_one(
    "CatBoost",
    cb_explainer,
    X_test_scaled,
    pred_cb_test,
    idx_wrong_1,
    idx_test,
    feature_cols,
)


### 3.4 LIME: 1–2 объекта (альтернативно)

Hint: сравните “локальные” признаки LIME с SHAP для тех же объектов — где выводы согласованы, а где расходятся?


In [ ]:
class_names = [str(c) for c in le.classes_]

lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=feature_cols,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)


def predict_proba_rf(x: np.ndarray) -> np.ndarray:
    return rf_for_xai.predict_proba(x)


def predict_proba_cb(x: np.ndarray) -> np.ndarray:
    return cb_for_xai.predict_proba(x)


def explain_with_lime(model_name: str, predict_fn, row_idx: int):
    pos = int(np.where(idx_test == row_idx)[0][0])
    x = X_test_scaled[pos]

    exp = lime_explainer.explain_instance(
        x,
        predict_fn,
        num_features=10,
        top_labels=1,
    )

    print(f"LIME ({model_name}) — idx={row_idx}")
    display(exp.as_list(label=exp.available_labels()[0]))


# 1 correct + 1 wrong
explain_with_lime("RF", predict_proba_rf, idx_correct_1)
explain_with_lime("RF", predict_proba_rf, idx_wrong_1)

explain_with_lime("CatBoost", predict_proba_cb, idx_correct_1)
explain_with_lime("CatBoost", predict_proba_cb, idx_wrong_1)


# VIII. Выводы (шаблон для студента)

- **Сравнение моделей**: 
  - 
  - 

- **Важные признаки по XAI** (Permutation / PDP / SHAP / LIME):
  - 
  - 

- **Общие ошибки моделей**:
  - 
  - 

- **Гипотезы** (почему модели ошибаются на пересечении ошибок):
  - 
  - 
